Stage 3 - Federated Learning for Cross-Country Iris Recognition (SwinV2 Tiny)

In [ ]:
import os, json, random, time, pathlib
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
import copy


import numpy as np
from PIL import Image


os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from sklearn.metrics import roc_auc_score, roc_curve

try:
    import timm
except Exception as e:
    timm = None
    print("timm is not installed. Install with: pip install timm")

# ---------------------- Global Config ----------------------
SEED = 42 
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ---- Determinism (set once, globally for the kernel session) ----
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# warn_only=True: SwinV2's relative-position-bias gather has no deterministic CUDA
# backward kernel; warn_only keeps it from crashing while pinning everything else.
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

INPUT_SIZE   = (64, 512)    # (H, W)
BATCH_SIZE   = 32
EMBED_DIM    = 256
MARGIN       = 0.3
TEMPERATURE  = 0.07
CLIP_NORM    = 5.0

# PK
P_IDENTITIES = 16
K_IMAGES     = 2

# Paths
RESULTS_ROOT = r"C:\Users\awais\OneDrive\Desktop\Thesis\Eye_Results"
SPLITS_DIR   = r"C:\Users\awais\OneDrive\Desktop\Thesis\splits\Eyes\FIHR"
MODELS_ROOT  = r"C:\Users\awais\OneDrive\Desktop\Thesis\Eye_Results"
FL_MODELS_ROOT = os.path.join(MODELS_ROOT, "FL_Stage3")
os.makedirs(FL_MODELS_ROOT, exist_ok=True)

DATASETS = {
    "Pakistan": r"C:\Users\awais\OneDrive\Desktop\Thesis\Pakistan\PakIris_Normalized_Aug",
    "China":    r"C:\Users\awais\OneDrive\Desktop\Thesis\China\CASIA-Iris-Interval_Normalized_Aug",
    "Czech":    r"C:\Users\awais\OneDrive\Desktop\Thesis\Czech\CzechIris_Normalized_Enhanced_FIHR",
    "India":    r"C:\Users\awais\OneDrive\Desktop\Thesis\India\IITD_Normalized",
    "Iraq":     r"C:\Users\awais\OneDrive\Desktop\Thesis\Iraq\AMF_Normalized_Enhanced_FIHR",
    "Malaysia": r"C:\Users\awais\OneDrive\Desktop\Thesis\Malaysia\MMU_Normalized_Enhanced_FIHR",
    "Iran":    r"C:\Users\awais\OneDrive\Desktop\Thesis\Iran\Iris_Normalized_Enhanced_FIHR"
}

def log(msg, level="INFO", dataset=None, model=None):
    prefix = f"[{level}]"
    if dataset and model: prefix += f" [{dataset} | {model}]"
    elif dataset:         prefix += f" [{dataset}]"
    elif model:           prefix += f" [{model}]"
    print(f"{prefix} {msg}")

Cell 2 - JSON train/val/test splits & mean/std

In [ ]:
def load_json_split(dataset_name: str, dataset_root: str):
    """
    Loads train/val/test split from JSON in SPLITS_DIR.
    Keeps your original structure.
    """
    fp = os.path.join(SPLITS_DIR, f"{dataset_name.lower()}_split.json")
    if not os.path.exists(fp):
        raise FileNotFoundError(fp)
    with open(fp, 'r', encoding='utf-8') as f:
        data = json.load(f)

    def finalize(lst):
        items = []
        for itm in lst:
            if isinstance(itm, (list, tuple)):
                path, lbl = itm[0], itm[1] if len(itm) > 1 else None
            elif isinstance(itm, dict):
                path = itm.get('path') or itm.get('image') or next(iter(itm.values()))
                lbl  = itm.get('label')
            else:
                path, lbl = itm, None
            abs_path = path if os.path.isabs(path) else os.path.join(dataset_root, path)
            if lbl is None:
                lbl = pathlib.Path(abs_path).parent.name
            items.append((abs_path, lbl))
        return items

    train = finalize(data.get('train', []))
    val   = finalize(data.get('val', []))
    test  = finalize(data.get('test', []))

    classes = sorted({l for _, l in train + val + test})
    log(f"Loaded {dataset_name}: train={len(train)}, val={len(val)}, test={len(test)}, classes={len(classes)}",
        "SPLIT", dataset_name)
    return {'train': train, 'val': val, 'test': test, 'classes': classes}

def estimate_mean_std(items: List[Tuple[str,str]], max_samples: int = 512):
    sel = items if len(items) <= max_samples else random.sample(items, max_samples)
    m = 0.0; s2 = 0.0; n = 0
    for p,_ in sel:
        with Image.open(p) as im:
            arr = np.asarray(im.convert('L'), dtype=np.float32) / 255.0
        m_batch = arr.mean(); s2_batch = arr.var(); k = arr.size
        new_n = n + k
        delta = m_batch - m
        m = m + delta * (k / new_n)
        s2 = (n*s2 + k*s2_batch + (delta**2)*n*k/new_n) / new_n
        n = new_n
    return float(m), float(np.sqrt(max(s2, 1e-8)))

Cell 3 - Siamese Train Dataset, PKSampler, IndexDataset

In [ ]:
from torchvision import transforms

class SiameseTrainSet(Dataset):
    """Grayscale 64x512 with augmentations for metric learning."""
    def __init__(self, items, label_to_idx, mean, std):
        self.items = items
        self.label_to_idx = label_to_idx
        self.mean, self.std = mean, std

        self.by_lbl: Dict[str, List[int]] = {}
        for i, (_, l) in enumerate(items):
            self.by_lbl.setdefault(l, []).append(i)

        self.tx = transforms.Compose([
            transforms.Grayscale(1),
            transforms.Resize(INPUT_SIZE),
            transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
            transforms.ColorJitter(brightness=0.3, contrast=0.3),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize([self.mean], [self.std]),
            transforms.RandomErasing(p=0.3, scale=(0.02, 0.1), ratio=(0.3, 3.3)),
        ])

    def __len__(self): return len(self.items)

    def load(self, idx: int):
        p, l = self.items[idx]
        img = Image.open(p).convert('L')
        x = self.tx(img)  # (1,64,512)
        return x, self.label_to_idx[l]

class PKSampler(Sampler[List[int]]):
    """Yield batches as P identities x K images indices."""
    def __init__(self, items, P=16, K=2, shuffle=True):
        self.items = items
        self.P, self.K = P, K
        self.shuffle = shuffle
        self.by_lbl: Dict[str, List[int]] = {}
        for i, (_, l) in enumerate(items):
            self.by_lbl.setdefault(l, []).append(i)
        self.labels = [l for l, idxs in self.by_lbl.items() if len(idxs) >= K]

    def __iter__(self):
        labels = self.labels.copy()
        if self.shuffle: random.shuffle(labels)
        for i in range(0, len(labels), self.P):
            grp = labels[i:i+self.P]
            if len(grp) < self.P: break
            idxs = []
            for l in grp:
                pool = self.by_lbl[l]
                sel = random.sample(pool, self.K) if self.shuffle else pool[:self.K]
                idxs.extend(sel)
            yield idxs

    def __len__(self):
        return max(0, len(self.labels)//self.P)

class IndexDataset(Dataset):
    """Returns just an index i; used with PKSampler."""
    def __init__(self, n): self.n = n
    def __len__(self): return self.n
    def __getitem__(self, i): return int(i)

Cell 4 - Metric-learning losses + ROC/EER evaluation (with interpolation)

In [ ]:
def pairwise_sim(z):
    z = F.normalize(z, dim=1)
    return z @ z.t()

def batch_hard_triplet_loss(z, y, margin=MARGIN):
    y = y.view(-1,1)
    S = pairwise_sim(z)
    D = torch.clamp(2 - 2*S, min=0.0)
    M = (y==y.t())
    pos = (D * M).where(M, torch.tensor(-1., device=D.device)).max(1).values
    neg = (D + 1e6*M.float()).min(1).values
    return F.relu(pos - neg + margin).mean()

def supervised_contrastive_loss(z, y, temperature=TEMPERATURE):
    z = F.normalize(z, dim=1)
    S = z @ z.t() / temperature
    y = y.view(-1,1)
    mask = (y==y.t()).float()
    logits_mask = torch.ones_like(mask) - torch.eye(mask.shape[0], device=mask.device)
    mask = mask * logits_mask
    log_prob = S - torch.logsumexp(S * logits_mask - 1e9*(1-logits_mask), dim=1, keepdim=True)
    pos_count = mask.sum(1).clamp_min(1.0)
    return (-(mask * log_prob).sum(1) / pos_count).mean()


# ---------- Interpolated EER and TAR@FAR helpers ----------

def _eer_from_fpr_tpr_interp(fpr, tpr):
    """
    EER via interpolation between the two points where fpr-fnr changes sign.
    Falls back to closest point if no sign change.
    """
    fnr = 1.0 - tpr
    diff = fpr - fnr

    # If never crosses zero, fall back to nearest point
    if np.all(diff >= 0) or np.all(diff <= 0):
        i = int(np.nanargmin(np.abs(diff)))
        return float(max(fpr[i], fnr[i]))

    # indices where sign changes between i and i+1
    idx = np.where(np.sign(diff[:-1]) != np.sign(diff[1:]))[0]
    i0 = int(idx[0])
    i1 = i0 + 1

    # linear interpolation on diff -> 0
    d0, d1 = diff[i0], diff[i1]
    t = d0 / (d0 - d1 + 1e-12)

    x0, x1 = fpr[i0], fpr[i1]
    eer = x0 + t * (x1 - x0)
    return float(eer)


def _tar_at_far_interp(fpr, tpr, far):
    """
    TAR at given FAR using linear interpolation on (fpr, tpr).
    """
    fpr = np.asarray(fpr)
    tpr = np.asarray(tpr)

    if fpr.size == 0:
        return float("nan")

    if far <= fpr[0]:
        return float(tpr[0])
    if far >= fpr[-1]:
        return float(tpr[-1])

    idx1 = int(np.searchsorted(fpr, far))
    idx0 = idx1 - 1

    x0, x1 = fpr[idx0], fpr[idx1]
    y0, y1 = tpr[idx0], tpr[idx1]

    if x1 == x0:
        return float(y0)

    alpha = (far - x0) / (x1 - x0)
    tar = y0 + alpha * (y1 - y0)
    return float(tar)


@torch.no_grad()
def compute_embeddings(model, items, mean, std, batch_size=BATCH_SIZE):
    """
    Evaluation pipeline:
    - grayscale
    - resize to (64, 512)
    - normalize with dataset-specific mean/std
    - L2-normalized embeddings from the FL model
    """
    tx = transforms.Compose([
        transforms.Grayscale(1),
        transforms.Resize(INPUT_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([mean], [std]),
    ])

    class _EvalSet(Dataset):
        def __init__(self, items):
            self.items = items
        def __len__(self):
            return len(self.items)
        def __getitem__(self, i):
            p, l = self.items[i]
            img = Image.open(p).convert("L")
            x = tx(img)
            return x, l

    pin = (DEVICE.type == "cuda")
    loader = DataLoader(
        _EvalSet(items),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=pin,
    )

    model.eval()
    feats, labels = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=pin)
        z = model(xb).cpu()
        feats.append(z)
        labels.extend(yb)

    return torch.cat(feats).numpy(), np.array(labels)


def evaluate_embeddings(feats, labels):
    """
    Given embeddings and identity labels, computes:
    - ROC-AUC
    - EER (interpolated)
    - TAR@FAR=0.01 and 0.001 (interpolated)
    """
    if len(labels) < 2:
        return {
            "roc_auc": float("nan"),
            "eer": float("nan"),
            "tar1": float("nan"),
            "tar01": float("nan"),
        }

    # cosine similarities of L2-normalized embeddings
    Z = torch.tensor(feats, dtype=torch.float32)
    Z = F.normalize(Z, dim=1)
    S = (Z @ Z.t()).numpy()

    m = labels.reshape(-1, 1) == labels.reshape(1, -1)
    iu = np.triu_indices_from(S, 1)
    y_true = m[iu].astype(np.uint8)
    y_score = S[iu]

    if len(np.unique(y_true)) < 2:
        return {
            "roc_auc": float("nan"),
            "eer": float("nan"),
            "tar1": float("nan"),
            "tar01": float("nan"),
        }

    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc  = float(roc_auc_score(y_true, y_score))
    eer  = _eer_from_fpr_tpr_interp(fpr, tpr)
    tar1 = _tar_at_far_interp(fpr, tpr, 0.01)
    tar01= _tar_at_far_interp(fpr, tpr, 0.001)

    return {
        "roc_auc": auc,
        "eer": eer,
        "tar1": tar1,
        "tar01": tar01,
    }

Cell 5 - SwinV2 Tiny Siamese (timm)

In [ ]:
class SwinV2_Tiny_Siamese(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        if timm is None:
            raise RuntimeError("timm is required for SwinV2_Tiny_Siamese. Install via pip.")
        m = timm.create_model(
            'swinv2_tiny_window8_256',  # as used in centralized training
            pretrained=True,
            num_classes=0,
            in_chans=1,
            img_size=INPUT_SIZE,
        )
        feat_dim = m.num_features
        self.backbone = m
        self.proj = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.GELU(),
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, embed_dim),
        )
    def forward(self, x):
        f = self.backbone(x)
        z = self.proj(f)
        return F.normalize(z, p=2, dim=1)

def swin_model_fn():
    return SwinV2_Tiny_Siamese(embed_dim=EMBED_DIM)

Cell 6 - FL configs (ONLY FHIR with polynomial-learned weights)

In [ ]:
from dataclasses import dataclass, field
import random
import numpy as np
import torch

# ---------------- Training ----------------
@dataclass
class FLTrainConfig:
    rounds: int = 200
    local_epochs: int = 1
    eval_every: int = 5
    patience: int = 20
    fp16_comms: bool = True

# ---------------- Optimizers (client) ----------------
@dataclass
class FLOptimConfig:
    lr: float = 1e-4
    weight_decay: float = 1e-4
    betas: tuple = (0.9, 0.999)

# ---------------- Loss ----------------
@dataclass
class FLLossConfig:
    supcon_weight: float = 0.5
    margin: float = MARGIN
    temperature: float = TEMPERATURE

# ---------------- FHIR + POLY settings ----------------
@dataclass
class FHIRPolyConfig:
    # Warmup: run OG-FHIR for 30-50 rounds to collect training data
    warmup_rounds: int = 50

    # Polynomial regression (degree 2)
    poly_degree: int = 2
    ridge_alpha: float = 1e-2  # regularization

    # how many (round,client) samples before first fit
    min_fit_samples: int = 80   # 7 clients * ~12 eval steps ~ 84

    # score stability
    score_floor: float = 1e-6

# ---------------- OG-FHIR weights (warmup only) ----------------
@dataclass
class FHIRWeightConfig:
    alpha_size: float    = 0.35   # a log(1 + N_k)
    beta_div: float      = 0.25   # b (q_k / N_k)
    gamma_quality: float = 0.25   # g Q_k
    delta_rarity: float  = 0.15   # d (1 / sqrt(q_k))

@dataclass
class FLExperimentConfig:
    train: FLTrainConfig = field(default_factory=FLTrainConfig)
    optim: FLOptimConfig = field(default_factory=FLOptimConfig)
    loss: FLLossConfig   = field(default_factory=FLLossConfig)

    # Only one strategy now
    strategy: str = "fhir_poly"   # <- ONLY strategy used

    # Warmup OG-FHIR weights + polynomial learner config
    fhir: FHIRWeightConfig = field(default_factory=FHIRWeightConfig)
    poly: FHIRPolyConfig   = field(default_factory=FHIRPolyConfig)

def set_global_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

Cell 7 - Minimal server state (ONLY FHIR)

In [ ]:
from dataclasses import dataclass
import copy
import torch

@dataclass
class ServerState:
    global_weights: dict
    round: int = 0

class Strategy:
    def init_state(self, model_fn):
        model = model_fn()
        return ServerState(global_weights=copy.deepcopy(model.state_dict()))

def build_strategy(name: str) -> Strategy:
    # only one supported strategy now
    if name != "fhir_poly":
        raise ValueError(f"Unsupported strategy: {name}. Use 'fhir_poly' only.")
    return Strategy()

Cell 8 - Prepare FL client data (splits, label maps, mean/std)

In [ ]:
from collections import namedtuple
import copy

FLClientData = namedtuple("FLClientData", ["name", "train_items", "val_items", "test_items",
                                           "label_to_idx", "mean", "std"])

def prepare_fl_clients():
    splits = {d: load_json_split(d, r) for d, r in DATASETS.items()}
    mean_std_cache = {}
    clients_data = {}

    for ds_name, split in splits.items():
        label_to_idx = {l: i for i, l in enumerate(split['classes'])}

        if ds_name not in mean_std_cache:
            m, s = estimate_mean_std(split['train'])
            mean_std_cache[ds_name] = (m, s)
            log(f"[FL] mean/std for {ds_name} = ({m:.3f}, {s:.3f})",
                "FL-NORM", ds_name, "SwinV2_Tiny_Siamese")
        mean, std = mean_std_cache[ds_name]

        clients_data[ds_name] = FLClientData(
            name=ds_name,
            train_items=split['train'],
            val_items=split['val'],
            test_items=split['test'],
            label_to_idx=label_to_idx,
            mean=mean,
            std=std
        )
    return clients_data

fl_clients_data = prepare_fl_clients()

Cell 9 - FLClient class (local training loop)

In [ ]:
class FLClient:
    def __init__(self, client_data: FLClientData,
                 cfg: FLExperimentConfig,
                 model_fn,
                 prox_mu: float = 0.0):
        self.data = client_data
        self.cfg = cfg
        self.model_fn = model_fn
        self.prox_mu = prox_mu

        self.train_set = SiameseTrainSet(
            items=self.data.train_items,
            label_to_idx=self.data.label_to_idx,
            mean=self.data.mean,
            std=self.data.std
        )

    def _prox_term(self, model, global_weights):
        """FedProx / FHIR-Prox proximal regularization term."""
        if self.prox_mu <= 0.0:
            return torch.tensor(0., device=DEVICE)

        prox = torch.tensor(0., device=DEVICE)
        for (n, p) in model.named_parameters():
            if p.requires_grad:
                prox = prox + torch.sum((p - global_weights[n].to(p.device)) ** 2)
        return 0.5 * self.prox_mu * prox

    def train_one_round(self, global_weights: dict):
        device = DEVICE
        model = self.model_fn().to(device)
        model.load_state_dict(copy.deepcopy(global_weights), strict=True)
        model.train()

        opt_cfg = self.cfg.optim
        opt = torch.optim.AdamW(
            model.parameters(),
            lr=opt_cfg.lr,
            weight_decay=opt_cfg.weight_decay,
            betas=opt_cfg.betas
        )

        total_loss = 0.0
        total_batches = 0

        for _ in range(self.cfg.train.local_epochs):
            pk_sampler = PKSampler(
                self.data.train_items,
                P=P_IDENTITIES,
                K=K_IMAGES,
                shuffle=True,
            )

            # IMPORTANT: keep this single-process (num_workers=0)
            # to avoid DataLoader worker crashes on Windows / notebooks.
            train_loader = DataLoader(
                IndexDataset(len(self.data.train_items)),
                batch_sampler=pk_sampler,
                num_workers=0,      # <- no multiprocessing workers
                pin_memory=False,   # GPU still works; this only affects host->device copies
            )

            for batch_idx in train_loader:
                xb, yb = [], []
                for idx in batch_idx.tolist():
                    x, y = self.train_set.load(int(idx))
                    xb.append(x)
                    yb.append(y)

                x = torch.stack(xb).to(device)
                y = torch.tensor(yb, dtype=torch.long).to(device)

                opt.zero_grad(set_to_none=True)
                z = model(x)

                # Metric-learning losses
                l_trip = batch_hard_triplet_loss(
                    z, y,
                    margin=self.cfg.loss.margin
                )
                l_sup = supervised_contrastive_loss(
                    z, y,
                    temperature=self.cfg.loss.temperature
                )

                loss = l_trip + self.cfg.loss.supcon_weight * l_sup

                # FedProx / FHIR-Prox proximal term (0 if mu == 0)
                loss = loss + self._prox_term(model, global_weights)

                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
                opt.step()

                total_loss += float(loss)
                total_batches += 1

        avg_loss = total_loss / max(1, total_batches)
        updated = copy.deepcopy(model.state_dict())

        # Compress updates to fp16 for communication if enabled
        if self.cfg.train.fp16_comms:
            for k in updated:
                updated[k] = updated[k].to(torch.float16).cpu()

        return updated, avg_loss

Cell 10 - Global evaluation on val/test of each client

In [ ]:
def evaluate_global_on_clients(model_fn, global_state_dict, fl_clients_data: dict, split_type="val"):
    results = {}
    for name, cdata in fl_clients_data.items():
        items = cdata.val_items if split_type == "val" else cdata.test_items
        model = model_fn().to(DEVICE)
        model.load_state_dict(global_state_dict, strict=True)
        feats, labels = compute_embeddings(model, items, cdata.mean, cdata.std)
        metrics = evaluate_embeddings(feats, labels)
        results[name] = metrics
    return results

Baselines - FedAvg, FedProx, FedYogi, FedAdam, FedNova(SGD), MOON, SCAFFOLD

All baselines reuse the SAME local training as FedHAT (AdamW + triplet/supcon on the PK
sampler, identical augs) and the SAME validation-macro-EER checkpoint selection. They
differ ONLY in the server-side aggregation / correction, which isolates the effect of the
FL strategy.

NOTE: FedNova and SCAFFOLD were derived for local SGD. Here the local optimizer is held
fixed at AdamW for a controlled comparison, so their step-normalization / control-variate
corrections sit on top of identical local training. This is a deliberate, defensible choice
(it keeps the comparison fair to FedHAT, which also trains locally with AdamW), but it is
off the original theory - state it in the paper.

FAIRNESS: tune each baseline's key hyperparameter (FedProx mu, FedYogi eta/tau, SCAFFOLD
eta_g) on validation, exactly as you tuned your own method, BEFORE concluding "FedHAT beats
X". An untuned baseline is not a baseline.

In [ ]:
# ============================================================
# Generalized local trainer (shared by all baselines)
#   - reuses FedHAT losses / PK sampler / augmentations
#   - optional FedProx proximal term
#   - optional SCAFFOLD control-variate correction
#   - returns local step count (needed by FedNova)
# ============================================================
def local_train_generalized(client, global_weights, *, prox_mu=0.0,
                            scaffold=False, c_global=None, c_local=None,
                            local_opt="adamw", sgd_lr=1e-2, sgd_momentum=0.0,
                            moon=False, moon_mu=1.0, moon_temp=0.5,
                            global_sd=None, prev_sd=None):
    """
    Returns:
        updated_state_dict  (fp16 if cfg.train.fp16_comms else fp32, on CPU)
        avg_loss
        n_steps             (local optimizer steps this round)
        c_local_new         (dict over trainable params; only for SCAFFOLD, else None)
    SCAFFOLD: corrected gradient  g <- g + (c_global - c_local)  before each opt.step().
    """
    device = DEVICE
    cfg = client.cfg
    model = client.model_fn().to(device)
    model.load_state_dict(copy.deepcopy(global_weights), strict=True)
    model.train()

    if local_opt == "sgd":
        opt = torch.optim.SGD(model.parameters(), lr=sgd_lr, momentum=sgd_momentum,
                              weight_decay=cfg.optim.weight_decay)
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=cfg.optim.lr,
                                weight_decay=cfg.optim.weight_decay, betas=cfg.optim.betas)

    # MOON: frozen global + previous-local models for the contrastive term
    gm = pm = None
    if moon:
        gm = client.model_fn().to(device)
        gm.load_state_dict(global_sd if global_sd is not None else global_weights, strict=True)
        gm.eval()
        for _p in gm.parameters(): _p.requires_grad_(False)
        if prev_sd is not None:                       # no previous model in round 1 -> skip contrast
            pm = client.model_fn().to(device)
            pm.load_state_dict(prev_sd, strict=True)
            pm.eval()
            for _p in pm.parameters(): _p.requires_grad_(False)

    if scaffold:
        cg = {n: c_global[n].to(device) for n in c_global}
        ci = {n: c_local[n].to(device)  for n in c_local}

    total_loss, n_steps = 0.0, 0
    for _ in range(cfg.train.local_epochs):
        pk = PKSampler(client.data.train_items, P=P_IDENTITIES, K=K_IMAGES, shuffle=True)
        loader = DataLoader(IndexDataset(len(client.data.train_items)),
                            batch_sampler=pk, num_workers=0, pin_memory=False)
        for batch_idx in loader:
            xb, yb = [], []
            for idx in batch_idx.tolist():
                x, y = client.train_set.load(int(idx)); xb.append(x); yb.append(y)
            x = torch.stack(xb).to(device)
            y = torch.tensor(yb, dtype=torch.long).to(device)

            opt.zero_grad(set_to_none=True)
            z = model(x)
            l_trip = batch_hard_triplet_loss(z, y, margin=cfg.loss.margin)
            l_sup  = supervised_contrastive_loss(z, y, temperature=cfg.loss.temperature)
            loss = l_trip + cfg.loss.supcon_weight * l_sup

            if prox_mu > 0.0:
                prox = torch.tensor(0.0, device=device)
                for nm, p in model.named_parameters():
                    if p.requires_grad:
                        prox = prox + torch.sum((p - global_weights[nm].to(device)) ** 2)
                loss = loss + 0.5 * prox_mu * prox

            if moon and (pm is not None):
                with torch.no_grad():
                    z_glob = gm(x)          # frozen global representation
                    z_prev = pm(x)          # frozen previous-local representation
                # z, z_glob, z_prev are already L2-normalized by the model -> dot == cosine
                pos = (z * z_glob).sum(dim=1) / moon_temp
                neg = (z * z_prev).sum(dim=1) / moon_temp
                moon_logits = torch.stack([pos, neg], dim=1)            # positive = global (index 0)
                moon_labels = torch.zeros(z.size(0), dtype=torch.long, device=device)
                loss = loss + moon_mu * F.cross_entropy(moon_logits, moon_labels)

            loss.backward()

            if scaffold:
                for nm, p in model.named_parameters():
                    if (p.grad is not None) and (nm in cg):
                        p.grad.add_(cg[nm] - ci[nm])     # g + (c - c_i)

            nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            opt.step()
            total_loss += float(loss); n_steps += 1

    avg_loss = total_loss / max(1, n_steps)
    updated = copy.deepcopy(model.state_dict())

    c_local_new = None
    if scaffold:
        lr = cfg.optim.lr
        denom = max(1, n_steps) * lr
        c_local_new = {}
        with torch.no_grad():
            for nm in c_local:
                w0 = global_weights[nm].to(device).float()
                wl = updated[nm].float()
                # SCAFFOLD option-II:  c_i^+ = c_i - c + (w_global - w_local) / (K * lr)
                c_local_new[nm] = (ci[nm] - cg[nm] + (w0 - wl) / denom).cpu()

    if cfg.train.fp16_comms:
        for k in updated: updated[k] = updated[k].to(torch.float16).cpu()
    else:
        for k in updated: updated[k] = updated[k].cpu()

    del model, opt
    if gm is not None: del gm
    if pm is not None: del pm
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return updated, avg_loss, n_steps, c_local_new

In [ ]:
# ============================================================
# Unified baseline training loop
#   strategy in {fedavg, fedprox, fedyogi, fednova, scaffold}
#   - server math runs on CPU/fp32 (same as FedHAT aggregation)
#   - same eval cadence + macro-EER best-checkpoint selection + early stopping
# ============================================================
SERVER_OPTS = {
    "fedavg":   {},
    "fedprox":  {"mu": 1e-2},                                  # tune: {1e-3, 1e-2, 1e-1, 1}
    "fedyogi":  {"eta": 1e-2, "beta1": 0.9, "beta2": 0.99, "tau": 1e-3},  # tune eta, tau
    "fedadam":  {"eta": 1e-2, "beta1": 0.9, "beta2": 0.99, "tau": 1e-3},  # FedOpt server Adam; tune eta
    "fednova":  {},
    "fednova_sgd": {"lr": 1e-2, "momentum": 0.0},             # FAITHFUL FedNova: vanilla local SGD; tune lr
    "moon":     {"mu": 1.0, "temp": 0.5},                     # MOON model-contrastive; tune mu (try 1, 5)
    "scaffold": {"eta_g": 1.0},                               # 1.0 is standard for full participation
}

def train_federated_baseline(model_fn, fl_clients_data, cfg, strategy, sopt=None):
    assert strategy in SERVER_OPTS, f"unknown strategy {strategy}"
    sopt = dict(SERVER_OPTS[strategy]) if sopt is None else dict(sopt)

    set_global_seed(SEED)
    state = ServerState(global_weights=copy.deepcopy(model_fn().state_dict()))
    clients = [FLClient(cd, cfg, model_fn, prox_mu=0.0) for cd in fl_clients_data.values()]
    client_names = [c.data.name for c in clients]
    N = len(clients)

    # data-size weights p_k = n_k / sum n
    n_k = {c.data.name: len(c.data.train_items) for c in clients}
    tot = float(sum(n_k.values()))
    p = {k: n_k[k] / tot for k in client_names}

    float_keys = [k for k, v in state.global_weights.items() if v.is_floating_point()]
    param_names = [nm for nm, pp in model_fn().named_parameters() if pp.requires_grad]

    # ---- per-strategy server state (CPU/fp32) ----
    if strategy in ("fedyogi", "fedadam"):
        m = {k: torch.zeros_like(state.global_weights[k], dtype=torch.float32) for k in float_keys}
        v = {k: torch.full_like(state.global_weights[k].float(), float(sopt["tau"]) ** 2) for k in float_keys}
    if strategy == "scaffold":
        named = dict(model_fn().named_parameters())
        c_global = {nm: torch.zeros_like(named[nm].detach()).cpu() for nm in param_names}
        c_locals = {cn: {nm: torch.zeros_like(c_global[nm]) for nm in param_names} for cn in client_names}
    if strategy == "moon":
        prev_locals = {cn: None for cn in client_names}   # previous-round local model per client (None in round 1)

    best_state = copy.deepcopy(state)
    best_round = 0
    best_macro = float("inf")
    bad = 0

    for r in range(cfg.train.rounds):
        rid = r + 1
        log(f"==== [BASE:{strategy}] Round {rid}/{cfg.train.rounds} ====", "FL")

        global_pre = {k: vv.to(torch.float32) for k, vv in state.global_weights.items()}

        updates, steps = [], []
        c_new_by = {}
        for c in clients:
            if strategy == "scaffold":
                upd, lss, st, ci_new = local_train_generalized(
                    c, global_pre, scaffold=True,
                    c_global=c_global, c_local=c_locals[c.data.name])
                c_new_by[c.data.name] = ci_new
            elif strategy == "moon":
                upd, lss, st, _ = local_train_generalized(
                    c, global_pre, moon=True,
                    moon_mu=float(sopt["mu"]), moon_temp=float(sopt["temp"]),
                    global_sd=global_pre, prev_sd=prev_locals[c.data.name])
                prev_locals[c.data.name] = {k: vv.detach().clone() for k, vv in upd.items()}
            elif strategy == "fednova_sgd":
                upd, lss, st, _ = local_train_generalized(
                    c, global_pre, local_opt="sgd",
                    sgd_lr=float(sopt["lr"]), sgd_momentum=float(sopt.get("momentum", 0.0)))
            else:
                prox_mu_val = float(sopt.get("mu", 0.0)) if strategy == "fedprox" else 0.0
                upd, lss, st, _ = local_train_generalized(c, global_pre, prox_mu=prox_mu_val)
            upd = {k: vv.to(torch.float32) for k, vv in upd.items()}
            updates.append(upd); steps.append(st)
            log(f"Client {c.data.name}: avg_loss={lss:.4f}, steps={st}", "FL-CLIENT")

        # ---- aggregate (CPU/fp32); non-float keys copied through unchanged ----
        new_global = {k: global_pre[k].clone() for k in global_pre}

        if strategy in ("fedavg", "fedprox", "moon"):
            for k in float_keys:
                acc = torch.zeros_like(global_pre[k])
                for cn, upd in zip(client_names, updates):
                    acc += p[cn] * upd[k]
                new_global[k] = acc

        elif strategy in ("fednova", "fednova_sgd"):
            tau_eff = sum(p[cn] * steps[i] for i, cn in enumerate(client_names))
            for k in float_keys:
                agg = torch.zeros_like(global_pre[k])
                for i, (cn, upd) in enumerate(zip(client_names, updates)):
                    agg += (p[cn] / max(1, steps[i])) * (upd[k] - global_pre[k])
                new_global[k] = global_pre[k] + tau_eff * agg

        elif strategy == "fedyogi":
            b1, b2, eta, tau = sopt["beta1"], sopt["beta2"], sopt["eta"], sopt["tau"]
            for k in float_keys:
                delta = torch.zeros_like(global_pre[k])
                for cn, upd in zip(client_names, updates):
                    delta += p[cn] * (upd[k] - global_pre[k])
                m[k] = b1 * m[k] + (1.0 - b1) * delta
                d2 = delta * delta
                v[k] = v[k] - (1.0 - b2) * d2 * torch.sign(v[k] - d2)
                new_global[k] = global_pre[k] + eta * m[k] / (torch.sqrt(v[k]) + tau)

        elif strategy == "fedadam":
            b1, b2, eta, tau = sopt["beta1"], sopt["beta2"], sopt["eta"], sopt["tau"]
            for k in float_keys:
                delta = torch.zeros_like(global_pre[k])
                for cn, upd in zip(client_names, updates):
                    delta += p[cn] * (upd[k] - global_pre[k])
                m[k] = b1 * m[k] + (1.0 - b1) * delta
                v[k] = b2 * v[k] + (1.0 - b2) * (delta * delta)   # standard Adam EMA (only diff from Yogi)
                new_global[k] = global_pre[k] + eta * m[k] / (torch.sqrt(v[k]) + tau)

        elif strategy == "scaffold":
            eta_g = float(sopt.get("eta_g", 1.0))
            for k in float_keys:
                avg_delta = torch.zeros_like(global_pre[k])
                for cn, upd in zip(client_names, updates):
                    avg_delta += (upd[k] - global_pre[k])
                new_global[k] = global_pre[k] + (eta_g / N) * avg_delta
            # control variate update:  c <- c + (1/N) * sum_i (c_i^+ - c_i)
            for nm in param_names:
                dc = torch.zeros_like(c_global[nm])
                for cn in client_names:
                    dc += (c_new_by[cn][nm] - c_locals[cn][nm])
                c_global[nm] = c_global[nm] + dc / N
            c_locals = {cn: c_new_by[cn] for cn in client_names}

        state.global_weights = new_global
        state.round += 1

        # ---- eval + checkpoint selection (identical to FedHAT) ----
        if rid % cfg.train.eval_every == 0:
            metrics = evaluate_global_on_clients(model_fn, state.global_weights,
                                                 fl_clients_data, split_type="val")
            for cn, mm in metrics.items():
                log(f"  {cn}: EER={mm['eer']:.4f}, TAR@1%={mm['tar1']:.4f}, "
                    f"TAR@0.1%={mm['tar01']:.4f}", "FL-METRIC")
            valid = [mm["eer"] for mm in metrics.values() if np.isfinite(mm["eer"])]
            macro = float(np.mean(valid)) if valid else float("inf")
            log(f"  [{strategy}] Macro-EER: {macro:.4f}", "FL-MACRO")

            if np.isfinite(macro) and macro + 1e-5 < best_macro:
                best_macro = macro; best_round = rid
                best_state = copy.deepcopy(state); bad = 0
                log(f"  New best macro-EER {best_macro:.4f} @ round {best_round}", "FL-BEST")
            else:
                bad += 1
                log(f"  No improvement. bad_rounds={bad}/{cfg.train.patience}", "FL-PATIENCE")
                if bad >= cfg.train.patience:
                    log(f"Early stopping @ {rid}. Best {best_macro:.4f} @ {best_round}", "FL-STOP")
                    break

    return best_state, {"best_round": best_round,
                        "best_macro_eer": best_macro,
                        "strategy": strategy}

In [ ]:
# ============================================================
# Runner - train + save each baseline (same schedule as FedHAT)
# ============================================================
import os

base_cfg = FLExperimentConfig()
base_cfg.train.rounds       = 250
base_cfg.train.local_epochs = 1
base_cfg.train.eval_every   = 1
base_cfg.train.patience     = 150
base_cfg.loss.supcon_weight = 0.5

SAVE_DIR = r"C:\Users\awais\OneDrive\Desktop\Thesis\Eye_Results\FL_Stage3"
os.makedirs(SAVE_DIR, exist_ok=True)

# pick which baselines to run this session
STRATEGIES = ["moon", "fedavg", "fedprox", "fedyogi", "fedadam", "fednova_sgd", "moon", "scaffold"]

baseline_summary = {}
for strat in STRATEGIES:
    log(f"################  TRAINING BASELINE: {strat}  (seed {SEED})  ################", "RUN")
    best_state, info = train_federated_baseline(
        model_fn=swin_model_fn,
        fl_clients_data=fl_clients_data,
        cfg=base_cfg,
        strategy=strat,
        sopt=SERVER_OPTS[strat],
    )
    path = os.path.join(SAVE_DIR, f"BASELINE_{strat}_seed{SEED}.pt")
    torch.save({
        "global_model": {k: v.cpu() for k, v in best_state.global_weights.items()},
        "strategy": strat,
        "server_opts": SERVER_OPTS[strat],
        "best_round": info["best_round"],
        "best_macro_eer": info["best_macro_eer"],
        "seed": SEED,
    }, path)
    baseline_summary[strat] = (info["best_round"], info["best_macro_eer"], path)
    print(f"[{strat}] best macro-EER {info['best_macro_eer']:.6f} @ round {info['best_round']}")
    print(f"   saved -> {path}")

print("\n==== Baseline summary (val macro-EER) ====")
for strat, (br, me, _) in baseline_summary.items():
    print(f"  {strat:9s}: {me:.6f}  (round {br})")